# 用 Qwen3-TTS 生成男女播音员语料
两位播音员各自朗读同一份 31,799 句清单，共生成 63,598 段录音。同一个 GPU 会话依次运行两种音色。将 `tiny-voca-mix-colab-31799.zip` 上传到 MyDrive 根目录，笔记本会自动提取文本与音素。可先勾选 `PREVIEW_ONLY`，试听男女声的中文、英文和混读各一句。

续跑时沿用相同的参考音和配置。完整生成后，训练笔记本会从同一个 Drive 目录读取两位播音员的录音。


In [ ]:
#@title 挂载 Drive，设置音色和并发
from google.colab import drive
from pathlib import Path
from collections import Counter
import hashlib
import json
import os
import subprocess
import sys

drive.mount("/content/drive")
ROOT = Path("/content/drive/MyDrive/tiny-voca-mix-qwen3-tts")

# 上传默认输入包后，两个笔记本均可独立恢复全量文本和音素。
import shutil
import zipfile
PACKAGE = ROOT.parent / 'tiny-voca-mix-colab-31799.zip'
if PACKAGE.is_file():
    (ROOT / 'inputs').mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(PACKAGE) as bundle:
        for name in ('corpus.jsonl', 'misaki.jsonl'):
            member = 'tiny-voca-mix-qwen3-tts/inputs/' + name
            dest = ROOT / 'inputs' / name
            temp = dest.with_suffix('.jsonl.part')
            with bundle.open(member) as src, temp.open('wb') as out:
                shutil.copyfileobj(src, out)
            temp.replace(dest)
MALE_DESCRIPTION = "一位成熟的男新闻播音员，音色沉稳浑厚，中低音区饱满。使用标准普通话，吐字清楚，停连自然，语速平稳，有央视新闻播报的沉着感。切换到英语时保持同一音色，重音自然。" #@param {type:"string"}
FEMALE_DESCRIPTION = "一位成熟女新闻播音员，声音清晰明亮，中音区圆润。使用标准普通话，吐字准确，语速平稳，停连自然。切换到英语时保持同一音色，重音自然。" #@param {type:"string"}
REFERENCE_TEXT = "各位听众，早上好。今天上午，连接两岸的新桥正式通车。 The new route will save commuters about twenty minutes." #@param {type:"string"}
VOICE_SEED = 42 #@param {type:"integer"}
CONCURRENCY = 32 #@param {type:"integer"}
PREVIEW_ONLY = False #@param {type:"boolean"}

if type(CONCURRENCY) is not int or CONCURRENCY < 1:
    raise ValueError("CONCURRENCY 必须是正整数")
if not all(s.strip() for s in (MALE_DESCRIPTION, FEMALE_DESCRIPTION)) or not REFERENCE_TEXT.strip() or not 0 <= VOICE_SEED < 2**32:
    raise ValueError("请输入音色描述、参考台词，以及 0 到 2**32-1 之间的随机种子。")

CORPUS = ROOT / "inputs/corpus.jsonl"
if not CORPUS.is_file():
    raise FileNotFoundError(f"请确认已将文本清单上传到 {CORPUS}")
def digest(data):
    return hashlib.sha256(data).hexdigest()

raw = CORPUS.read_bytes()
rows = [json.loads(line) for line in raw.splitlines() if line.strip()]
language_map = {"zh": "Chinese", "en": "English", "mixed": "Auto"}
if not rows or len({r["id"] for r in rows}) != len(rows):
    raise ValueError("清单为空或存在重复句子 ID")
import re
for row in rows:
    if (not re.fullmatch(r"[A-Za-z0-9_-]+", row["id"])
        or row["language"] not in language_map or not row["text"].strip()
        or row["split"] not in {"train", "validation", "test"}):
        raise ValueError(f"无效的清单条目 {row['id']}")
corpus_sha = digest(raw)
if corpus_sha != "582ad8a8b4a395bc63e4ad3714ef278ab760e1a126e290478f317cc2a1b5961a":
    raise ValueError("请使用输入包中的完整 31,799 句 corpus.jsonl")
print(f"{len(rows):,} 条", dict(Counter(r["language"] for r in rows)))
for lang in language_map:
    example = next((r["text"] for r in rows if r["language"] == lang), None)
    if example:
        print(lang, example)

def atomic_text(path, text):
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".part")
    tmp.write_text(text, encoding="utf-8")
    tmp.replace(path)

if MALE_DESCRIPTION.strip() == FEMALE_DESCRIPTION.strip():
    raise ValueError("请为两位播音员设置不同的音色描述")
voice_dirs = {}
for speaker, description in (("male", MALE_DESCRIPTION), ("female", FEMALE_DESCRIPTION)):
    voice = {
        "description": description, "reference_text": REFERENCE_TEXT, "reference_language": "Auto", "seed": VOICE_SEED,
        "design_revision": "5ecdb67327fd37bb2e042aab12ff7391903235d3", "qwen_tts": "0.1.1",
    }
    design = json.dumps(voice, ensure_ascii=False, sort_keys=True, indent=2)
    folder = ROOT / "voices" / digest(design.encode())[:16]
    atomic_text(folder / "design.json", design)
    voice_dirs[speaker] = folder
BASE_REVISION = "fd4b254389122332181a7c3db7f27e918eec64e3"
WORK = Path("/content/tiny-voca-qwen-runtime")
WORK.mkdir(exist_ok=True)


## 安装推理依赖
Python 环境装在 Colab 临时盘，可在新会话中重建。权重、下载缓存和合成数据均写入 Drive。


In [ ]:
#@title 配置 GPU 推理环境
import shutil
if not shutil.which("nvidia-smi") or subprocess.run(["nvidia-smi", "-L"]).returncode:
    raise RuntimeError("请在 Colab 的「代码执行程序 → 更改运行时类型」中选择 GPU，再运行本单元。")

env = os.environ.copy()
env.update(HF_HOME=str(ROOT / "cache/huggingface"), TORCH_HOME=str(ROOT / "cache/torch"),
           HF_HUB_DISABLE_XET="1", TOKENIZERS_PARALLELISM="false")
def run(*args):
    command = [str(arg) for arg in args]
    print("$", *command, flush=True)
    with subprocess.Popen(command, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                          text=True, errors="replace", bufsize=1) as process:
        for line in process.stdout:
            print(line.rstrip(), flush=True)
        status = process.wait()
    if status:
        raise subprocess.CalledProcessError(status, command)

gpu = subprocess.check_output(["nvidia-smi", "-i", "0", "--query-gpu=name,compute_cap,driver_version",
                               "--format=csv,noheader"], text=True).strip()
name, capability, driver = [value.strip() for value in gpu.split(",")]
wheel = "cu128" if int(capability.split(".")[0]) >= 10 else "cu126"
print(f"GPU {name} | 计算能力 {capability} | 驱动 {driver} | PyTorch {wheel}", flush=True)
if not shutil.which("sox"):
    run("apt-get", "update", "-qq")
    run("apt-get", "install", "-qy", "sox")
run(sys.executable, "-m", "pip", "install", "-q", "uv")
PYTHON = WORK / "venv/bin/python"
if not PYTHON.exists():
    run("uv", "venv", "--python", "3.11", WORK / "venv")
run("uv", "pip", "install", "--python", PYTHON, "--index-url", f"https://download.pytorch.org/whl/{wheel}",
    f"torch==2.8.0+{wheel}", f"torchaudio==2.8.0+{wheel}")
run("uv", "pip", "install", "--python", PYTHON, "qwen-tts==0.1.1", "gradio==5.49.1", "numpy==2.2.6")
run(PYTHON, "-c", """
import torch, torchaudio, qwen_tts
dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
x = torch.ones((32, 32), device="cuda", dtype=dtype)
assert (x @ x + 1).float().mean().item() == 33
print("CUDA 运算验证通过", torch.__version__, torch.version.cuda, torch.cuda.get_device_name(), dtype)
""")


## 准备推理程序
下面的单元把程序写入临时运行环境。输出 WAV 按语言和句子 ID 分目录，音频写完并验证后才使用正式文件名。


In [ ]:
%%writefile /content/tiny-voca-qwen-runtime/synthesize.py
import argparse
from collections import Counter
import gc
import hashlib
import json
import os
from pathlib import Path
import wave
import time

import numpy as np
import soundfile as sf
import torch
from huggingface_hub import snapshot_download
from qwen_tts import Qwen3TTSModel

RATE = 24000
LANGUAGE = {"zh": "Chinese", "en": "English", "mixed": "Auto"}


def load_model(root, kind, revision):
    name = f"Qwen3-TTS-12Hz-1.7B-{kind}"
    folder = root / "models" / name / revision
    # local_dir 把权重、分词器和可恢复的未完成下载一起放在 Drive。
    print(f"正在准备模型 {name}，权重目录 {folder}", flush=True)
    snapshot_download("Qwen/" + name, revision=revision, local_dir=folder,
                      allow_patterns=["*.json", "*.txt", "*.safetensors"], max_workers=2)
    dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
    print(f"正在加载模型到 GPU，精度 {dtype}", flush=True)
    return Qwen3TTSModel.from_pretrained(str(folder), device_map="cuda:0", dtype=dtype,
                                         attn_implementation="sdpa", local_files_only=True)


def wav_frames(path):
    # 验证 PCM 头和最后一帧，防止断写文件被跳过。
    try:
        with wave.open(str(path), "rb") as handle:
            frames = handle.getnframes()
            if (frames < 1 or handle.getframerate() != RATE or handle.getnchannels() != 1
                or handle.getsampwidth() != 2):
                return 0
            handle.setpos(frames - 1)
            return frames if len(handle.readframes(1)) == 2 else 0
    except (FileNotFoundError, wave.Error, EOFError):
        return 0


def save_wav(path, samples, rate):
    samples = np.asarray(samples)
    if rate != RATE or samples.ndim != 1 or not samples.size or not np.isfinite(samples).all():
        raise ValueError(f"无效的模型输出 {path.name}")
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(".wav.part")
    sf.write(tmp, samples, rate, format="WAV", subtype="PCM_16")
    if wav_frames(tmp) != samples.size:
        raise IOError(f"WAV 写入不完整 {tmp}")
    os.replace(tmp, path)


def design_voice(root, voice_dir):
    ref = voice_dir / "reference.wav"
    if wav_frames(ref):
        print("使用已保存的参考音", ref)
        return
    config = json.loads((voice_dir / "design.json").read_text())
    torch.manual_seed(config["seed"])
    model = load_model(root, "VoiceDesign", config["design_revision"])
    wavs, rate = model.generate_voice_design(text=config["reference_text"], language=config["reference_language"],
                                            instruct=config["description"])
    save_wav(ref, wavs[0], rate)


def audio_path(row):
    bucket = hashlib.sha256(row["id"].encode()).hexdigest()[:2]
    return Path("audio") / row["language"] / bucket / (row["id"] + ".wav")


def generation_limit(batch):
    # 模型每秒约生成 12.5 帧，按本批最长文本留出停顿、慢读的余量。
    return min(8192, max(128, 80 + 6 * max(len(row["text"]) for row in batch)))


def generate_batch(model, prompt, batch, limit, *, non_streaming=False):
    return model.generate_voice_clone(
        text=[r["text"] for r in batch], language=[LANGUAGE[r["language"]] for r in batch],
        voice_clone_prompt=prompt, max_new_tokens=limit, non_streaming_mode=non_streaming)


def synthesize(root, voice_dir, out, revision, corpus_sha, concurrency, preview):
    raw = (root / "inputs/corpus.jsonl").read_bytes()
    if hashlib.sha256(raw).hexdigest() != corpus_sha:
        raise ValueError("语料已变更，请从配置单元重新运行。")
    rows = [json.loads(line) for line in raw.splitlines() if line.strip()]
    if preview:
        rows = [next(r for r in rows if r["language"] == lang)
                for lang in LANGUAGE if any(r["language"] == lang for r in rows)]
    paths = {r["id"]: audio_path(r) for r in rows}
    completed = {}
    expected = {path: key for key, path in paths.items()}
    print(f"正在列出已有 WAV，目标 {len(rows)} 条，目录 {out}", flush=True)
    newest = None
    newest_stamp = (-1, "")
    def walk_error(error):
        raise error
    if (out / "audio").is_dir():
        for folder, _, files in os.walk(out / "audio", onerror=walk_error):
            relative = Path(folder).relative_to(out)
            for name in files:
                path = relative / name
                if path not in expected:
                    continue
                stat = (out / path).stat()
                # 本程序用 libsndfile 写标准 PCM16 单声道 WAV，44 字节头，每帧 2 字节。
                frames = (stat.st_size - 44) // 2
                if frames > 0:
                    completed[expected[path]] = frames
                stamp = (stat.st_mtime_ns, name)
                if stamp > newest_stamp:
                    newest, newest_stamp = path, stamp
    if newest is not None:
        frames = wav_frames(out / newest)
        if frames:
            completed[expected[newest]] = frames
        else:
            completed.pop(expected[newest], None)
        print(f"最近的录音 {newest.name}，{'校验通过' if frames else '需要重生成'}", flush=True)
    pending = [row for row in rows if row["id"] not in completed]

    print(f"本次目标 {len(rows)} 条，复用 {len(completed)} 条，待生成 {len(pending)} 条。", flush=True)
    if pending:
        model = load_model(root, "Base", revision)
        config = json.loads((voice_dir / "design.json").read_text())
        print("正在提取参考音特征", flush=True)
        prompt = model.create_voice_clone_prompt(ref_audio=str(voice_dir / "reference.wav"),
                                                ref_text=config["reference_text"])
        # 长度相近的句子放在同一个 GPU batch，减少等待最长句的时间。
        pending.sort(key=lambda r: len(r["text"]))
        deferred = Counter()
        reused = len(completed)
        offset, batch_size = 0, concurrency
        print(f"开始合成首批，batch={batch_size}", flush=True)
        started = last_report = time.monotonic()
        while offset < len(pending):
            batch = pending[offset:offset + batch_size]
            try:
                limit = generation_limit(batch)
                wavs, rate = generate_batch(model, prompt, batch, limit)
            except torch.cuda.OutOfMemoryError:
                if len(batch) == 1:
                    raise
                batch_size = max(1, len(batch) // 2)
                print(f"显存不足，并发降至 {batch_size}，重试当前批。", flush=True)
            else:
                if len(wavs) != len(batch):
                    raise RuntimeError("模型返回的音频数量与输入不符")
                for row, samples in zip(batch, wavs):
                    # 12 Hz 模型每帧 1920 个采样点。保守留两帧余量，避免把截断音频记为完成。
                    if len(samples) >= (limit - 2) * 1920:
                        retry_limit = min(8192, 2 * limit)
                        print(f"{row['id']} 接近 {limit} 步上限，单句重试", flush=True)
                        for attempt in range(3):
                            retry_wavs, rate = generate_batch(model, prompt, [row], retry_limit,
                                                              non_streaming=(attempt != 1))
                            samples = retry_wavs[0]
                            if len(samples) < (retry_limit - 2) * 1920:
                                break
                        else:
                            key = row["id"]
                            deferred[key] += 1
                            if deferred[key] > 2:
                                raise RuntimeError(f"{key} 多次重试仍触及上限，保留已完成录音")
                            pending.append(row)
                            print(f"{key} 移至队尾，先生成其余录音", flush=True)
                            continue
                    save_wav(out / paths[row["id"]], samples, rate)
                    completed[row["id"]] = len(samples)
                offset += len(batch)
                now = time.monotonic()
                if offset == len(batch) or now - last_report >= 30 or offset == len(pending):
                    added = len(completed) - reused
                    speed = added * 60 / max(now - started, 0.001)
                    print(f"合成 {len(completed)}/{len(rows)}，本次新增 {added}，"
                          f"batch={batch_size}，平均 {speed:.1f} 段/分钟", flush=True)
                    last_report = now
                continue
            gc.collect()
            torch.cuda.empty_cache()

    manifest = out / ("preview.jsonl" if preview else "corpus.jsonl")
    tmp = manifest.with_suffix(".jsonl.part")
    with tmp.open("w", encoding="utf-8") as stream:
        for row in rows:
            record = {**row, "audio": paths[row["id"]].as_posix(), "voice_id": voice_dir.name,
                      "sample_rate": RATE, "seconds": completed[row["id"]] / RATE}
            print(json.dumps(record, ensure_ascii=False), file=stream)
    os.replace(tmp, manifest)
    print("已完成", dict(Counter(r["language"] for r in rows)), manifest)


if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("stage", choices=["design", "synthesize"])
    parser.add_argument("root", type=Path)
    parser.add_argument("voice_dir", type=Path)
    parser.add_argument("--out", type=Path)
    parser.add_argument("--revision")
    parser.add_argument("--corpus-sha")
    parser.add_argument("--concurrency", type=int, default=2)
    parser.add_argument("--preview", action="store_true")
    args = parser.parse_args()
    if args.stage == "design":
        design_voice(args.root, args.voice_dir)
    else:
        synthesize(args.root, args.voice_dir, args.out, args.revision, args.corpus_sha, args.concurrency, args.preview)


## 设计并试听参考音
调整音色描述、参考台词或种子会创建新音色目录。点击播放器即可试听，然后运行下一节。


In [ ]:
from IPython.display import Audio, display
WORKER = WORK / "synthesize.py"
for speaker, folder in voice_dirs.items():
    run(PYTHON, "-u", WORKER, "design", ROOT, folder)
    print(speaker)
    display(Audio(filename=str(folder / "reference.wav")))


## 生成语料并导出清单
中文使用 Chinese，英文使用 English，混读句保持整句并使用 Auto。每条 WAV 为 24 kHz、16 位单声道。完整运行会写出带原始 ID、文本、划分、来源、相对音频路径和时长的 `corpus.jsonl`。试听模式写入 `preview.jsonl`。


In [ ]:
import math
import wave

def complete_male_run(out, recipe, rows):
    try:
        if json.loads((out / "recipe.json").read_text()) != recipe:
            return False
        expected = {row["id"]: row for row in rows}
        paths = set()
        with (out / "corpus.jsonl").open(encoding="utf-8") as manifest:
            for line in manifest:
                entry = json.loads(line)
                row = expected.pop(entry["id"])
                if not all(entry.get(k) == v for k, v in row.items()):
                    return False
                key = row["id"]
                path = Path("audio") / row["language"] / digest(key.encode())[:2] / (key + ".wav")
                seconds = entry["seconds"]
                if (entry["audio"] != path.as_posix() or entry["voice_id"] != recipe["voice"]
                        or entry["sample_rate"] != 24000 or not math.isfinite(seconds) or seconds <= 0):
                    return False
                paths.add(path)
    except (FileNotFoundError, ValueError, KeyError, TypeError, AttributeError):
        return False
    if expected or not paths:
        return False

    def walk_error(error):
        raise error

    actual = {Path(folder).relative_to(out) / name
              for folder, _, names in os.walk(out / "audio", onerror=walk_error)
              for name in names if name.endswith(".wav")} if (out / "audio").is_dir() else set()
    if actual != paths:
        return False
    latest = max(actual, key=lambda path: ((out / path).stat().st_mtime_ns, path.name))
    try:
        with wave.open(str(out / latest), "rb") as wav:
            if (wav.getnchannels(), wav.getsampwidth(), wav.getframerate()) != (1, 2, 24000) or wav.getnframes() < 1:
                return False
            wav.setpos(wav.getnframes() - 1)
            return len(wav.readframes(1)) == 2
    except (FileNotFoundError, EOFError, wave.Error):
        return False

outputs = {}
for speaker, VOICE_DIR in voice_dirs.items():
    reference_sha = digest((VOICE_DIR / "reference.wav").read_bytes())
    recipe = {"corpus_sha256": corpus_sha, "reference_sha256": reference_sha, "voice": VOICE_DIR.name,
              "base_revision": BASE_REVISION, "language_map": language_map, "format": "pcm16-24k-v1"}
    recipe_json = json.dumps(recipe, ensure_ascii=False, indent=2, sort_keys=True)
    OUT = ROOT / "runs" / digest(recipe_json.encode())[:16]
    if speaker == "male" and not PREVIEW_ONLY and complete_male_run(OUT, recipe, rows):
        outputs[speaker] = OUT
        print(f"男声清单与 {len(rows)} 个 WAV 路径匹配，最近一段校验通过。", flush=True)
        continue
    atomic_text(OUT / "recipe.json", recipe_json)
    print(f"开始 {speaker}，并发 {CONCURRENCY}，输出目录 {OUT}", flush=True)
    run(PYTHON, "-u", WORKER, "synthesize", ROOT, VOICE_DIR, "--revision", BASE_REVISION,
        "--corpus-sha", corpus_sha, "--out", OUT, "--concurrency", CONCURRENCY,
        *(["--preview"] if PREVIEW_ONLY else []))
    outputs[speaker] = OUT
if not PREVIEW_ONLY:
    atomic_text(ROOT / "announcers.json", json.dumps({k:v.name for k,v in outputs.items()}))


## 试听三类输出


In [ ]:
for speaker, OUT in outputs.items():
    print(speaker)
    manifest = OUT / ("preview.jsonl" if PREVIEW_ONLY else "corpus.jsonl")
    samples = {}
    with manifest.open(encoding="utf-8") as stream:
        for line in stream:
            record = json.loads(line)
            samples.setdefault(record["language"], record)
            if len(samples) == len(language_map):
                break
    for language, record in samples.items():
        print(language, record["text"])
        display(Audio(filename=str(OUT / record["audio"])))
    print("输出清单", manifest)
